# **Return of the Schema** for  a *General User Defined Knowledge Graph* 

## Path Definition Basic Elements

In [2]:
from rdflib import Graph, RDF, RDFS, OWL, Namespace
from urllib.parse import quote
from rdflib.namespace import split_uri
from rdflib.term import URIRef
from pathlib import Path
import pickle
import csv
import ast
import json
import sys
sys.path.append(str(Path.cwd().parent))
from kgsaf_jdex.utils.conventions.builtins import BUILTIN_URIS
import kgsaf_jdex.utils.conventions.paths as pc
from kgsaf_jdex.utils.modularization import SignatureModularizer, SchemaDecomposer 


def serialize(graph, path):
    graph.serialize(path.with_suffix(".xml"), format="xml")
    !/home/navis/robot/robot merge --input {path.with_suffix(".xml")} --output {path.with_suffix(".owl")}
    path.with_suffix(".xml").unlink()

In [3]:
REASONER = False
DATASET_FILE = "kg_consistent.owl"
DATASET_NAME = "kg_consistent"
DATASET_NAME = DATASET_NAME + "_reasoned" if REASONER else DATASET_NAME + "_base"
ROBOT_JAR = "/home/navis/robot/robot.jar"

home_path = Path().cwd().absolute()
dataset_path = home_path / "input" / DATASET_FILE
DATASET_PATH = home_path / "output" / f"{DATASET_NAME}"
 

(DATASET_PATH / "abox" / "splits").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "mappings").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "tbox").mkdir(parents=True, exist_ok=True)
(DATASET_PATH / "rbox").mkdir(parents=True, exist_ok=True)

In [4]:
from pathlib import Path

properties = [
    "SubClass",
    "EquivalentClass",
    "EquivalentObjectProperty",
    "InverseObjectProperties",
    "ObjectPropertyCharacteristic",
    "SubObjectProperty",
    "ObjectPropertyRange",
    "ObjectPropertyDomain",
    "ClassAssertion"
]

print("Axioms Generator:")
prop_string = ""
for p in properties:
    print(f"\t{p}")
    prop_string += " " + p



debug_path = DATASET_PATH / "debug.owl"
output_file = DATASET_PATH / "intermediate_kg.owl"

print("Base Path:", home_path)
print("Dataset:", dataset_path)

if REASONER:
    print("Running Reasoner on Target Ontology: Realization and Materialization")
    !java -Xmx20G -jar {ROBOT_JAR} reason -vvv \
        --reasoner HermiT \
        --input {dataset_path} \
        --output {output_file} \
        --axiom-generators "{prop_string}" \
        --remove-redundant-subclass-axioms false \
        --exclude-tautologies structural \
        --include-indirect true \
        -D {debug_path}
else:
    print("Running Reasoner on Target Ontology: Conversion to OWL Format")
    !java -Xmx20G -jar {ROBOT_JAR} merge -vvv \
        --input {dataset_path} \
        --output {output_file}

Axioms Generator:
	SubClass
	EquivalentClass
	EquivalentObjectProperty
	InverseObjectProperties
	ObjectPropertyCharacteristic
	SubObjectProperty
	ObjectPropertyRange
	ObjectPropertyDomain
	ClassAssertion
Base Path: /home/navis/dev/kg-saf/tutorial
Dataset: /home/navis/dev/kg-saf/tutorial/input/kg_consistent.owl
Running Reasoner on Target Ontology: Conversion to OWL Format
2026-01-20 14:54:39,693 DEBUG org.obolibrary.robot.IOHelper - Loading ontology /home/navis/dev/kg-saf/tutorial/input/kg_consistent.owl with catalog file null
2026-01-20 14:54:39,697 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading file META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2026-01-20 14:54:39,697 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/navis/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2026-01-20 14:54:39,697 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/

# ABOX Triple Cleaning: Object Properties Filtering

Removal of triples with individuals that are also classes

In [5]:
kg = Graph()
kg.parse(output_file)

<Graph identifier=N51977de86a0d4127831810c4cecfb8b7 (<class 'rdflib.graph.Graph'>)>

In [6]:
triples_graph = Graph()
for s,p,o in kg:
    if (s, RDF.type, OWL.NamedIndividual) in kg and (p, RDF.type, OWL.ObjectProperty) in kg and (o, RDF.type, OWL.NamedIndividual) in kg:
        triples_graph.add((s,p,o))

In [7]:
individuals = set()
predicates = set(triples_graph.predicates())

print(f"Initial Dataset: {len(triples_graph)} triples found (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual) Found")
print(f"Initial Dataset: {len(predicates)} predicates (OWL.ObjectProperty) Found")

print("Analyzing SUBJECTS")
for ind in triples_graph.subjects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")

print("Analyzing OBJECTS")
for ind in triples_graph.objects(None, None):
    if (ind, RDF.type, OWL.Class) not in kg:
        individuals.add(ind)
    else:
        print(f"Individual {ind} removed from dataset. Defined as Class")


print(f"Dataset will contain a total of {len(individuals)} individuals (OWL.NamedIndividual)")

print(f"Removing Triples...")

for s,p,o in triples_graph:
    if (s not in individuals) or (o not in individuals):
        triples_graph.remove((s,p,o))

print(f"Dataset will contain a total of {len(triples_graph)} triples (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual)")

print("Serializing Intermediate File")
triples_graph.serialize(DATASET_PATH / "obj_prop_assertion.nt", format="nt")

with open(DATASET_PATH / "obj_prop_assertion.tsv", "w") as f:
    for s, p, o in triples_graph:
        f.write(f"{s}\t{p}\t{o}\n")
print("Done")


Initial Dataset: 76943 triples found (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual) Found
Initial Dataset: 25 predicates (OWL.ObjectProperty) Found
Analyzing SUBJECTS
Analyzing OBJECTS
Dataset will contain a total of 29767 individuals (OWL.NamedIndividual)
Removing Triples...
Dataset will contain a total of 76943 triples (OWL.NamedIndividual, OWL.ObjectProperty, OWL.NamedIndividual)
Serializing Intermediate File


/home/navis/.pyenv/versions/oml/lib/python3.12/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Done


In [ ]:
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.triples.splitting import CoverageSplitter
import numpy as np


triples = TriplesFactory.from_path(DATASET_PATH / "obj_prop_assertion.tsv")

entity_mappings = {v:k for k,v in triples.entity_id_to_label.items()}
relation_mappings = {v:k for k,v in triples.relation_id_to_label.items()}

train, valid, test = triples.split(
    ratios=[0.85, 0.05, 0.1],
    random_state=42,
    method=CoverageSplitter(),      
)

train_clean = TriplesFactory.from_labeled_triples(
    triples=train.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

valid_clean = TriplesFactory.from_labeled_triples(
    triples=valid.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

test_clean = TriplesFactory.from_labeled_triples(
    triples=test.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

print(train_clean)
print(test_clean)
print(valid_clean)


from pykeen.triples.leakage import unleak

train_unleak, valid_unleak, test_unleak = unleak(
    train_clean,
    *[valid_clean, test_clean],
    n=None,
    minimum_frequency=0.97
    )

print(train_unleak)
print(test_unleak)
print(valid_unleak)


targets = [
    (DATASET_PATH / "abox/splits/train", train_unleak.triples),
    (DATASET_PATH / "abox/splits/valid", valid_unleak.triples),
    (DATASET_PATH / "abox/splits/test", test_unleak.triples)
]

for path, split in targets:
    out_graph = Graph()
    for triple in split:
        s = URIRef(triple[0])
        p = URIRef(triple[1])
        o = URIRef(triple[2])
        out_graph.add((URIRef(s), URIRef(p), URIRef(o)))

    out_graph.serialize(path.with_suffix(".nt"), format="nt")

!cat {DATASET_PATH}/abox/splits/*.nt > {DATASET_PATH}/abox/obj_prop_assertions.nt

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=65401)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=7695)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=3847)


Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=65401)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=7695)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=3847)


/home/navis/.pyenv/versions/oml/lib/python3.12/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [9]:
out_graph = Graph()

for ind in individuals:
    out_graph.add((ind, RDF.type, OWL.NamedIndividual))

serialize(out_graph, DATASET_PATH / "abox" / "individuals")
del out_graph

### [BASE] RDF Lib Class Assertions

In [12]:
class_assertions_graph = Graph()

for ind in individuals:
    for ca in set(kg.objects(ind, RDF.type)) - BUILTIN_URIS:
        if (ca, RDF.type, OWL.Class) in kg:
            class_assertions_graph.add((ind, RDF.type, ca))
        else:
            print(f"Not a Class {ca}")

serialize(class_assertions_graph, DATASET_PATH / "abox" / "class_assertions")

# TBOX and RBOX Extraction

In [16]:
seed_obj_props = predicates
print("Seed Object Properties", len(seed_obj_props))

seed_classes = set(class_assertions_graph.objects(None, RDF.type))
print("Seed Classes", len(seed_classes))

Seed Object Properties 25
Seed Classes 53


In [19]:
modularizer = SignatureModularizer(kg, seed_classes | seed_obj_props)
out_graph = modularizer.modularize(verbose=True)

serialize(out_graph, DATASET_PATH / "ontology")

Processing https://w3id.org/italia/onto/CLV/City
Processing https://w3id.org/italia/onto/SM/hasContactPoint
Processing https://apuliatravel.org/td/SeaCave
Processing https://w3id.org/italia/onto/AccessCondition/OpeningHoursSpecification
Processing https://apuliatravel.org/td/Cave
Processing https://w3id.org/italia/onto/CLV/hasNumber
Processing https://w3id.org/italia/onto/CLV/hasStreetToponym
Processing https://apuliatravel.org/td/NaturalArea
Processing https://apuliatravel.org/td/EntertainmentBusiness
Processing https://apuliatravel.org/td/Tower
Processing https://apuliatravel.org/td/EcoMuseum
Processing https://apuliatravel.org/td/Farmhouse
Processing https://w3id.org/italia/onto/AccessCondition/hasAccessCondition
Processing https://w3id.org/italia/onto/SM/hasReview
Processing https://w3id.org/italia/onto/CLV/Geometry
Processing https://apuliatravel.org/td/Ravine
Processing https://w3id.org/italia/onto/SM/Rating
Processing https://w3id.org/italia/onto/CLV/hasAddressComponent
Processi

In [22]:
onto_graph = Graph()
onto_graph.parse(DATASET_PATH / "ontology.owl")

decomposer = SchemaDecomposer(onto_graph)
rbox_graph, taxonomy_graph, schema_graph = decomposer.decompose(verbose=True)

serialize(rbox_graph, DATASET_PATH / "rbox" / "roles")
serialize(taxonomy_graph, DATASET_PATH / "tbox" / "taxonomy")
serialize(schema_graph, DATASET_PATH / "tbox" / "schema")


Processing https://w3id.org/italia/onto/TI/isTemporalEntityOf
Processing https://w3id.org/italia/onto/CLV/issuedBy
Processing https://w3id.org/italia/onto/SM/hasContactPoint
Processing https://w3id.org/italia/onto/SM/isTelephoneOf
Processing https://w3id.org/italia/onto/CLV/hasProvince
Processing https://w3id.org/italia/onto/CLV/hasGeometry
Processing https://w3id.org/italia/onto/SM/hasUserAccount
Processing https://w3id.org/italia/onto/CLV/hasNumber
Processing https://w3id.org/italia/onto/CLV/hasStreetToponym
Processing https://w3id.org/italia/onto/SM/hasOnlineContactPoint
Processing https://w3id.org/italia/onto/SM/isWebSiteOf
Processing https://w3id.org/italia/onto/CLV/hasAddress
Processing https://w3id.org/italia/onto/POI/hasPOICategory
Processing https://w3id.org/italia/onto/SM/isEmailTypeOf
Processing https://w3id.org/italia/onto/CLV/hasAdressinTime
Processing https://w3id.org/italia/onto/SM/hasCreator
Processing https://w3id.org/italia/onto/AccessCondition/hasAccessCondition
Proc

# Final Ontology and Knowledge Graph

In [ ]:
(DATASET_PATH / "intermediate_kg.owl").unlink()
(DATASET_PATH / "obj_prop_assertion.nt").unlink()
(DATASET_PATH / "obj_prop_assertion.tsv").unlink()

In [27]:
!/home/navis/robot/robot merge \
--input  {DATASET_PATH / "ontology.owl"} \
--input  {DATASET_PATH / "abox" / "individuals.owl"} \
--input {DATASET_PATH / "abox" / "triples.nt"} \
--input {DATASET_PATH / "abox" / "class_assertions.owl"} \
--output {DATASET_PATH / "knowledge_graph.owl"}

In [29]:
class TSVConverter:
    def __init__(
        self,
        path: str,
    ):

        self.p_data = dict()
        self.base_path = Path(path).resolve().absolute()

    def convert(
        self,
        triples: bool = True,
        splits: bool = True,
    ):

        if triples:
            self.p_data["triples"] = (
                self.preprocess_triples(self.base_path / "abox/obj_prop_assertions.nt"),
                self.base_path / "abox/obj_prop_assertions.tsv",
            )

        if splits:
            self.p_data["train"] = (
                self.preprocess_triples(self.base_path / pc.RDF_TRAIN),
                self.base_path / pc.TRAIN,
            )
            self.p_data["test"] = (
                self.preprocess_triples(self.base_path / pc.RDF_TEST),
                self.base_path / pc.TEST,
            )
            self.p_data["valid"] = (
                self.preprocess_triples(self.base_path / pc.RDF_VALID),
                self.base_path / pc.VALID,
            )


    def serialize(self):
        for key, values in self.p_data.items():
            obj = values[0]
            path = values[1]

            with open(path, "w") as f:
                if key in ["triples", "train", "valid", "test"]:
                    f.write(obj)

    def preprocess_triples(self, path):
        triples = Graph()
        triples.parse(path)
        out_str = ""
        for s, p, o in triples:
            out_str += f"{str(s)}\t{str(p)}\t{str(o)}\n"
        return out_str
    

print(f"\t Converting Dataset {DATASET_PATH.name}")
processor = TSVConverter(DATASET_PATH)
processor.convert()
processor.serialize()

	 Converting Dataset kg_consistent_base


In [30]:
import kgsaf_jdex.utils.conventions.paths as pc
from kgsaf_jdex.utils.conversion import OWLConverter

print(f"\tProcessing Dataset {DATASET_PATH.name}")
processor = OWLConverter(DATASET_PATH)
processor.preprocess(verbose=False)
processor.serialize()

	Processing Dataset kg_consistent_base
Processing Dataset at /home/navis/dev/kg-saf/tutorial/output/kg_consistent_base
Processing Taxonomy
Processing Class Assertions
Processing Object Property Hierarchy
Processing Object Property Domain and Range
